# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Two paper findings and my methodology questions

### Finding 1 — "Fresh Content Always Outperforms" is nuanced

The research paper reports that recently refreshed content can show stronger Health Scores, especially when the content is already comprehensive. The paper reports a 12.8-point Health Score difference between recently updated 3,500+ word content and content that was 181+ days old.

My methodology question is: does the analysis show that freshness itself causes better performance, or could stronger content be more likely to be selected for refreshing? In other words, could selection or other confounding factors explain part of the observed difference?

The paper uses observational comparisons, so this finding should be interpreted as an observed association rather than proof that refreshing a page causes the improvement.

### Finding 2 — "AI-Generated Content Is Penalized" is debunked

The paper reports that its analysis did not support a blanket penalty associated with AI-generated content after comparing model cohorts within age tiers.

My methodology question is: does controlling for content age and comparing model cohorts adequately account for other differences between the content groups, such as topic, editing quality, or publishing period?

Because this is observational data rather than an experiment, the result supports a measured association in the analyzed portfolio but does not prove that AI generation itself causes or prevents a ranking outcome.

### Why these questions matter

Both questions are about whether the validation design supports the strength of the claim. I am not trying to reject the findings. I am checking what the data and methodology can actually support.

In [56]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
# Load starter dataset
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/hafizahmadadilaiengineer/flyrank-ml-internship.git

repo_root = Path("flyrank-ml-internship")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")



print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Unique content pages:", df["content_id"].nunique())

Dataset shape: (30000, 44)
Unique clients: 32
Unique content pages: 30000


In [57]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Declining pages:", df["is_declining_label"].sum())
print("Non-declining pages:", (df["is_declining_label"] == 0).sum())

Declining pages: 16262
Non-declining pages: 13738


In [58]:
from sklearn.model_selection import train_test_split

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))

Training clients: 25
Test clients: 7

Training rows: 26581
Test rows: 3419


In [59]:
# Week-5 feature configuration

target = "is_declining_label"

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

print("Target:", target)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

Target: is_declining_label
Numeric features: 29
Categorical features: 11
Total features: 40


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [60]:
# ML-09 — Before vs After validation audit
# BEFORE = random row split
# AFTER  = client-grouped split

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# ---------------------------------------------------------
# Helper function
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()


# ---------------------------------------------------------
# Feature definitions — same as Week 5
# ---------------------------------------------------------

feature_columns = numeric_features + categorical_features


# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# ---------------------------------------------------------
# Random Forest — same model as Week 5
# ---------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)


# =========================================================
# BEFORE — Random row split
# =========================================================

random_train, random_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[target]
)

X_random_train = random_train[feature_columns]
X_random_test = random_test[feature_columns]

y_random_train = random_train[target]
y_random_test = random_test[target]


random_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

random_pipeline.fit(
    X_random_train,
    y_random_train
)

random_scores = random_pipeline.predict_proba(
    X_random_test
)[:, 1]

random_precision50 = precision_at_k(
    y_random_test.reset_index(drop=True),
    pd.Series(random_scores),
    k=50
)

random_ap = average_precision_score(
    y_random_test,
    random_scores
)


# Check client overlap
random_client_overlap = (
    set(random_train["client_id"])
    & set(random_test["client_id"])
)


# =========================================================
# AFTER — Client-grouped split
# =========================================================

X_group_train = train_df[feature_columns]
X_group_test = test_df[feature_columns]

y_group_train = train_df[target]
y_group_test = test_df[target]


grouped_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

grouped_pipeline.fit(
    X_group_train,
    y_group_train
)

grouped_scores = grouped_pipeline.predict_proba(
    X_group_test
)[:, 1]

grouped_precision50 = precision_at_k(
    y_group_test.reset_index(drop=True),
    pd.Series(grouped_scores),
    k=50
)

grouped_ap = average_precision_score(
    y_group_test,
    grouped_scores
)


# Check client overlap
grouped_client_overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)


# =========================================================
# BEFORE vs AFTER table
# =========================================================

validation_results = pd.DataFrame({
    "Validation": [
        "Random row split",
        "Client-grouped split"
    ],
    "Train rows": [
        len(random_train),
        len(train_df)
    ],
    "Test rows": [
        len(random_test),
        len(test_df)
    ],
    "Train clients": [
        random_train["client_id"].nunique(),
        train_df["client_id"].nunique()
    ],
    "Test clients": [
        random_test["client_id"].nunique(),
        test_df["client_id"].nunique()
    ],
    "Client overlap": [
        len(random_client_overlap),
        len(grouped_client_overlap)
    ],
    "Precision@50": [
        random_precision50,
        grouped_precision50
    ],
    "Average Precision": [
        random_ap,
        grouped_ap
    ]
})

display(validation_results)

,Validation,Train rows,Test rows,Train clients,Test clients,Client overlap,Precision@50,Average Precision
0,Random row split,24000,6000,32,31,31,1.0,0.881710
1,Client-grouped split,26581,3419,25,7,0,0.8,0.872244


### Before vs after interpretation

The random row split allows pages from the same client to appear in both the training and test sets. In this dataset, the random split therefore has client overlap.

The client-grouped split keeps complete clients on one side of the evaluation. My grouped split contains 25 training clients and 7 test clients with zero client overlap.

I treat the client-grouped result as the more appropriate validation estimate because the model is evaluated on clients whose pages were not used during training.

The comparison is not intended to show that the random split is a better or worse model. Its purpose is to demonstrate how the validation design can change the measured result. For this project, client-grouped validation better matches the risk of learning client-specific patterns.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [61]:
# ML-09 — Leakage audit of final Week-5 feature set

feature_columns = numeric_features + categorical_features

audit_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

leakage_audit = pd.DataFrame({
    "Column": audit_columns,
    "Used_as_feature": [
        col in feature_columns
        for col in audit_columns
    ]
})

display(leakage_audit)

print("\nFinal feature count:", len(feature_columns))

print("\nExcluded from model:")
for col in [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]:
    print("-", col)

unexpected = (
    set(["content_id", "client_id", "trend_direction", "trend_pct"])
    & set(feature_columns)
)

print(
    "\nUnexpected leakage columns in feature vector:",
    unexpected
)

,Column,Used_as_feature
0,is_declining_label,False
1,trend_direction,False
2,trend_pct,False
3,content_id,False
4,client_id,False



Final feature count: 40

Excluded from model:
- content_id
- client_id
- trend_direction
- trend_pct

Unexpected leakage columns in feature vector: set()


### Leakage audit conclusion

The final Week-5 feature vector does not contain `trend_direction`, `trend_pct`, `content_id`, or `client_id`. This prevents direct leakage from the label-derived fields and prevents pseudonymous identifiers from being used as predictive features.

However, there is an important limitation in the target definition. `is_declining_label` is created from `trend_direction`, and `trend_direction` is calculated from the current 30-day versus previous 30-day impression change.

Therefore, the label is a current-window proxy rather than a future observed outcome.

The model can be interpreted as ranking or classifying pages associated with the current declining proxy. It should not be described as proof that the model predicts future decline.

A stronger future version would build features from a prior window and define the target using a separate future window.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [62]:
claim_audit = pd.DataFrame({
    "Original claim": [
        "Random Forest is better than the Week-4 baseline.",
        "The model identifies pages that need to be refreshed.",
        "The model predicts declining pages."
    ],

    "Safer claim": [
        "Random Forest had the highest Average Precision, but the Week-4 baseline was better at the specific top-50 review task.",

        "The model ranks pages for content review based on their association with the declining proxy label.",

        "The model classifies pages according to the current declining proxy defined from trend_direction; this experiment does not establish future decline prediction."
    ]
})

display(claim_audit)

,Original claim,Safer claim
0,Random Forest is better than the Week-4 baseline.,Random Forest had the highest Average Precisio...
1,The model identifies pages that need to be ref...,The model ranks pages for content review based...
2,The model predicts declining pages.,The model classifies pages according to the cu...


## Claim rewrite

My strongest result should be described as an observed and measured result from the client-grouped test split, not as a general statement about future performance.

The Week-4 baseline achieved Precision@50 of 0.72, while Random Forest achieved Precision@50 of 0.46 and Average Precision of 0.6392. Therefore, the baseline was stronger for the specific top-50 review decision in this experiment, while Random Forest had the strongest overall ranking metric among the tested methods.

I can describe the output as decision-support for prioritizing pages for human review.

I cannot claim that the model proves which pages will decline in the future, that refreshing a recommended page will cause recovery, or that the model explains Google's ranking algorithm.

The main limitation is that the starter `is_declining_label` is a current-window proxy derived from `trend_direction`, rather than a future observed outcome.

In [63]:
# Real failure examples from the Week-5 Random Forest

grouped_predictions = test_df.copy()

grouped_predictions["model_score"] = grouped_scores
grouped_predictions["prediction"] = (
    grouped_predictions["model_score"] >= 0.50
).astype(int)

false_positives = grouped_predictions[
    (grouped_predictions["prediction"] == 1) &
    (grouped_predictions[target] == 0)
].sort_values("model_score", ascending=False)

false_negatives = grouped_predictions[
    (grouped_predictions["prediction"] == 0) &
    (grouped_predictions[target] == 1)
].sort_values("model_score", ascending=True)

print("Top false positives:")
display(
    false_positives[
        [
            "content_id",
            target,
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(5)
)

print("\nTop false negatives:")
display(
    false_negatives[
        [
            "content_id",
            target,
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(5)
)

Top false positives:


,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update
2357,content_8f1409b2674e,0,0.864064,209,0.00,20.0,104
20736,content_41baf0722ad9,0,0.844643,3115,0.00,12.8,104
1517,content_816d77e36e14,0,0.839858,208,0.48,25.3,104
28718,content_ef6e7d7cfe15,0,0.839670,264,0.00,22.2,104
11061,content_0b47dae0c7f9,0,0.835441,1191,0.00,23.1,103



Top false negatives:


,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update
12240,content_dc5c6e103bbb,1,0.345069,26208,0.74,5.4,15
11155,content_31aee330eb1c,1,0.355906,18561,0.34,4.8,15
19863,content_9fd1fb307b32,1,0.380822,65555,0.31,7.7,15
16089,content_216a54125880,1,0.394608,282,0.00,58.1,104
20477,content_63f07c4a37e7,1,0.402355,36201,0.30,6.4,15


### Overall validation conclusion

The validation audit strengthened the interpretation of my Week-5 results. The client-grouped split provides a more appropriate estimate than a random row split because it prevents pages from the same client appearing in both training and testing.

The feature audit found no direct use of the label-derived fields `trend_direction` or `trend_pct`, and identifiers such as `content_id` and `client_id` were not used as predictive features.

The failure analysis showed that the Random Forest can produce high scores for non-declining pages and low scores for some observed declining pages, including pages with substantial impressions and relatively strong search positions.

Most importantly, the starter `is_declining_label` is a current-window proxy derived from `trend_direction`, rather than a future observed outcome. Therefore, the Week-5 model should be described as decision-support for ranking pages associated with the declining proxy, not as a proven predictor of future decline or refresh success.

## Self-check

- [x] I reviewed two findings from the FlyRank research paper.
- [x] I asked a constructive methodology question about each finding.
- [x] I compared a naive random row split with a client-grouped split.
- [x] I verified that the client-grouped split has zero client overlap.
- [x] I audited the final Week-5 feature vector for leakage.
- [x] I excluded `trend_direction` and `trend_pct` from model features.
- [x] I excluded `content_id` and `client_id` from model features.
- [x] I identified the starter declining label as a proxy rather than a future outcome.
- [x] I inspected real Random Forest false positives and false negatives in Week 5.
- [x] I rewrote claims that were stronger than the evidence.
- [x] I used observed, measured, directional, and decision-support language.
- [x] I did not claim that refreshing a page causes recovery.
- [x] I did not claim to explain Google's ranking algorithm.
- [x] I ran the notebook from top to bottom with no errors.
- [x] I committed the completed notebook to `work/notebooks/w06_validation_audit.ipynb`.